# Can I run these?

*Checks your credentials for real, then tells you which notebook in this repo you can run right now.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omnifroodle/couchbase_notebooks/blob/main/notebooks/00_check_setup.ipynb)
[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/omnifroodle/couchbase_notebooks?quickstart=1)

**Claim.** Every notebook here declares what it needs. This one checks what you have and joins the two.
**Result.** A per-notebook verdict: ready, or blocked and on what.
**Requires.** nothing
**Read** ~1 min · **Run** ~1 min · **Cost** ~$0.00

Start here. Nothing below sets anything up — it only looks — so it is safe to run
repeatedly, and it is the fastest way to find out whether a missing key or a paused
cluster is about to waste twenty minutes of your time.

The checks are **live**: a real connection to Couchbase, a real call to your model
provider. Confirming that a key is *present* is worthless — a revoked key looks
identical to a working one until something tries to use it.

In [1]:
# --- Setup. Works in a local checkout and on Colab. -------------------------
import os
import pathlib
import subprocess
import sys

# Cloned on Colab, where there is no local checkout. Override to test a fork.
REPO_URL = os.environ.get("CBNB_REPO_URL", "https://github.com/omnifroodle/couchbase_notebooks")

try:
    import cbnb
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    root = next((p for p in [here, *here.parents] if (p / "cbnb" / "__init__.py").exists()), None)
    if root is None:
        # Colab: clone the repo so the committed datasets come with it.
        subprocess.check_call(["git", "clone", "--depth", "1", "--quiet", REPO_URL, "cbnb-repo"])
        root = pathlib.Path("cbnb-repo").resolve()
    sys.path.insert(0, str(root))
    import cbnb

# This notebook deliberately requires nothing: it has to run when nothing works.
settings = cbnb.bootstrap(requires=[])

cbnb ready (local, autoreload on). Couchbase: cb.***.cloud.couchbase.com (bucket 'demos') | LLM: nanogpt/z-ai/glm-5.3 | embeddings: local


## 1. What this environment can do

Each row is one **capability** — a coarse, named thing a notebook can ask for. The
checks below actually exercise them, so this takes a few seconds and may download the
embedding model the first time.

In [2]:
from cbnb import readiness

results = readiness.report()  # live checks: real connection, real API call

for r in results:
    print(f"[{'ok' if r.ok else '--'}] {r.name:<17} {r.detail}")
    for line in (r.fix.splitlines() if not r.ok else []):
        print(f"{'':<21}{line}")

[--] api-embeddings    NanoGPT rejected the API key (HTTP 401).
                     The key came from NANOGPT_API_KEY, via an environment variable.
                     
                     Fix it for this session:  cbnb.update_setting('NANOGPT_API_KEY')
                       then re-run the cell that creates the LLM.
                     Fix it for good: put the right value for NANOGPT_API_KEY in the repo's .env file (or wherever you export it), then restart the kernel.
[ok] couchbase         connected to cb.***.cloud.couchbase.com, bucket 'demos'
[ok] dataset-download  dataset host reachable (HTTP 200)
[--] llm               NanoGPT rejected the API key (HTTP 401).
                     The key came from NANOGPT_API_KEY, via an environment variable.
                     
                     Fix it for this session:  cbnb.update_setting('NANOGPT_API_KEY')
                       then re-run the cell that creates the LLM.
                     Fix it for good: put the right value for 

## 2. Which notebooks you can run

Every notebook states its requirements in its own title cell and setup call. This reads
those declarations straight from the files — nothing is registered anywhere, so a
notebook added tomorrow appears here without this one being edited.

In [3]:
from cbnb import inventory

have = {r.name for r in results if r.ok}

for lab in inventory.labs():
    blocked = [c for c in lab.requires if c not in have]
    if not lab.declared_in:
        verdict = "?  requirements not declared"
    elif blocked:
        verdict = f"-- needs {', '.join(blocked)}"
    else:
        verdict = "ok ready to run"
    print(f"{verdict}")
    print(f"   {lab.path.name}  —  {lab.title}")
    if lab.tagline:
        print(f"   {lab.tagline[:88]}")
    print()

ok ready to run
   00_check_setup.ipynb  —  Can I run these?
   Checks your credentials for real, then tells you which notebook in this repo you can run

-- needs llm
   01_hypothetical_classification.ipynb  —  Don't classify. Hallucinate.
   Cheap LLM classification into a taxonomy too large to fit in a prompt — with Couchbase V



## 3. If something came back blocked

**A missing setting.** The `fix:` line above is specific to where you are running —
`.env` locally, the secrets panel on Colab, repository secrets on Codespaces. The names
themselves are all in [`.env.example`](../.env.example).

**A rejected key.** The provider answered and said no. That is a wrong, expired or
revoked key rather than a missing one, and re-pasting the same value will not help — check
it in your provider's dashboard first. To try a different one for the rest of this session
without editing any file:

```python
cbnb.update_setting("NANOGPT_API_KEY")   # or OPENAI_API_KEY, GROQ_API_KEY, ...
```

Then re-run the check above. That only lasts for the session; to keep it, put it where the
`fix:` line says.

**Couchbase unreachable.** Almost always one of three things: the cluster is paused, your
IP is not on its allow list, or you are using your Capella *login* rather than a database
access user. [`docs/capella-setup.md`](../docs/capella-setup.md) walks through all three.

**Nothing blocked?** Start with
[`01_hypothetical_classification.ipynb`](01_hypothetical_classification.ipynb).